# Spotify — `songs_with_attributes_and_lyrics.csv` Preprocessing

Loads the raw Spotify dataset from `data/raw/`, applies all cleaning and deduplication steps for a **lyrics-based unsupervised recommendation system**, and saves the result to `data/processed/spotify_clean.csv`.

**Constraints:** no NLP libraries · no stemming/lemmatisation · no stopword removal · preserve lyric structure (line breaks).

In [ ]:
# Bring in the tools we need:
# re        → for finding and replacing patterns in text (like removing punctuation)
# hashlib   → for creating a unique fingerprint from lyrics text
# Path      → makes file/folder paths easy to work with on any OS

# pandas    → the main library for loading and working with spreadsheet-like dataimport pandas as pd

import refrom pathlib import Path
import hashlib

In [ ]:
# Tell Python where our files live.
# ROOT is the folder this notebook is running from.
# RAW_CSV  → the original messy data we downloaded from Kaggle.
# OUT_CSV  → where we'll save the clean version after processing.
ROOT          = Path.cwd()
RAW_CSV       = ROOT / 'data' / 'raw' / 'songs_with_attributes_and_lyrics.csv'
PROCESSED_DIR = ROOT / 'data' / 'processed'
OUT_CSV       = PROCESSED_DIR / 'spotify_clean.csv'

print('output will be:', OUT_CSV)

# Double-check the raw file is actually there before we start.print('raw exists?   ', RAW_CSV.exists())

In [ ]:
# ── Reusable helper functions ─────────────────────────────────────────────
# These are small tools we'll call repeatedly during cleaning.

def ascii_ratio(text: str) -> float:
    # Checks how "English-friendly" a piece of text is.
    # Regular English characters (a-z, A-Z, digits, common punctuation) all
    # have an ASCII code below 128.  Chinese, Arabic, Cyrillic etc. do not.
    # We return a number between 0 and 1 — 1.0 means 100% standard characters.
    if not isinstance(text, str) or len(text) == 0:
        return 0.0
    return sum(ord(c) < 128 for c in text) / len(text)


def normalize_meta(s: str) -> str:
    # Cleans up a song title or artist name so that
    # "Bohemian Rhapsody (Remastered)" and "Bohemian Rhapsody" are treated
    # as the same song, preventing accidental duplicates.
    if not isinstance(s, str):
        return ''
    s = s.lower()                                                          # make everything lowercase
    s = re.sub(r'\([^)]*(live|remaster(ed)?).*?\)', '', s, flags=re.IGNORECASE)  # remove (live) / (remastered 2011) etc.
    s = re.sub(r'[\-\u2013\u2014]\s*live\s*$', '', s, flags=re.IGNORECASE)       # remove trailing "- live"
    s = re.sub(r'\([^)]*\)', '', s)                                        # drop any leftover parentheses
    s = re.sub(r'[^\w\s]', ' ', s)                                         # swap punctuation for a space
    return re.sub(r'\s+', ' ', s).strip()                                  # collapse multiple spaces into one


def normalize_lyrics(text: str) -> str:
    # Cleans up lyrics text so two songs with identical words but different
    # capitalisation/punctuation get caught as duplicates.
    # Importantly, we KEEP line breaks so verse/chorus structure is preserved.
    if not isinstance(text, str):
        return ''
    t = text.lower()                                            # all lowercase
    t = re.sub(r"[^\w\s']", ' ', t)                            # remove punctuation except apostrophes (don't → don t would lose meaning)
    lines = [re.sub(r'[ \t]+', ' ', line).strip()              # tidy spaces within each line ...
             for line in t.splitlines()]                        # ... but keep the line breaks themselves
    return '\n'.join(lines)



def md5_hash(s: str) -> str:    return hashlib.md5(s.encode('utf-8')).hexdigest()

    # Creates a short fixed-length "fingerprint" of a string.    # If two lyrics produce the same fingerprint they are identical — fast way to spot duplicates.

In [ ]:
# ── Step 1 · Load the raw data ────────────────────────────────────────────
# Read the CSV file into a pandas DataFrame (think of it like an Excel table).
# We check the file exists first so we get a helpful error message instead of a cryptic crash.
if not RAW_CSV.exists():
    raise FileNotFoundError(
        f'Raw CSV not found at {RAW_CSV}\n'
        'Place songs_with_attributes_and_lyrics.csv inside data/raw/ and re-run.'
    )


# low_memory=False tells pandas to read the whole file before deciding column typesprint(f'Loaded  : {len(df):,} rows  |  columns: {list(df.columns)}')

# — avoids misleading "mixed types" warnings on large files.df = pd.read_csv(RAW_CSV, low_memory=False)

In [ ]:
# ── Step 2 · Drop columns we do not need ─────────────────────────────────
# The dataset has lots of audio features (tempo, energy, danceability…) but
# our system is lyrics-only, so we throw all of that away and keep just:
#   id      → unique identifier for each song
#   name    → song title  (we'll rename it to 'title' for clarity)
#   artists → performer   (we'll rename it to 'artist')
#   lyrics  → the actual lyrics text — the heart of our project
REQUIRED = ['id', 'name', 'artists', 'lyrics']
missing  = [c for c in REQUIRED if c not in df.columns]

if missing:print(f'After column selection  : {len(df):,} rows')

    raise ValueError(f'Missing columns in CSV: {missing}')  # blow up early with a clear messagedf = df.rename(columns={'name': 'title', 'artists': 'artist'})

df = df[REQUIRED].copy()

In [ ]:
# ── Step 3 · Remove songs with missing or very short lyrics ───────────────
# Songs with no lyrics are useless to us — drop them.
# Songs with fewer than 30 words are probably just interludes or data errors;
# they would not give the clustering algorithm enough text to work with.
df = df[df['lyrics'].notna()].copy()                              # remove rows where lyrics field is blank

df['_word_count'] = df['lyrics'].astype(str).str.split().str.len()  # count how many words each song hasprint(f'After null/short filter : {len(df):,} rows')
df = df[df['_word_count'] >= 30].copy()                           # keep only songs with 30+ words

In [ ]:
# ── Step 4 · Keep English-language songs only ─────────────────────────────
# Our feature extraction is designed for English text, so foreign-language
# songs would add noise.  We use a simple trick: if 90%+ of the characters
# are standard ASCII (plain English letters/punctuation), it's probably English.

# No language-detection library needed — just maths.print(f'After English filter    : {len(df):,} rows')

df['_ascii'] = df['lyrics'].astype(str).map(ascii_ratio)  # score each song 0–1df = df[df['_ascii'] >= 0.90].copy()                      # keep only the high-scoring (likely English) ones

In [ ]:
# ── Step 5 · Clean up song titles and artist names ───────────────────────
# Before we can spot duplicates, we need titles/artists in a consistent format.
# "Stairway To Heaven (Remastered)" and "stairway to heaven" should match.
# normalize_meta() lowercases everything, strips live/remaster labels, and

# removes punctuation.  We store the cleaned version in separate helper columnsprint('Title / artist normalised.')

# (prefixed with _) so the originals are still available.df['_norm_artist'] = df['artist'].astype(str).map(normalize_meta)
df['_norm_title']  = df['title'].astype(str).map(normalize_meta)

In [ ]:
# ── Step 6 · Remove perfectly identical rows ─────────────────────────────
# If every single column in two rows is identical, one is a straight copy.
# drop_duplicates() keeps the first occurrence and removes the rest.
before = len(df)

df = df.drop_duplicates().copy()print(f'Removed exact duplicates        : {before - len(df):,}  →  {len(df):,} remain')

In [ ]:
# ── Step 7 · Remove duplicates that are the same song by a different name ─
# Two rows might have slightly different formatting but actually be the same
# song by the same artist (e.g. different versions in the catalogue).
# We build a combined key like "bohemian rhapsody_queen" and drop any row
# where we have already seen that key before.

df['_comp_key'] = df['_norm_title'] + '_' + df['_norm_artist']  # stick title and artist together as one IDprint(f'Removed composite-key duplicates: {before - len(df):,}  →  {len(df):,} remain')

before = len(df)df = df.drop_duplicates(subset=['_comp_key']).copy()             # keep only the first occurrence of each key

In [ ]:
# ── Step 8 · Clean lyrics text and remove songs with identical lyrics ─────
# First we clean the lyrics (lowercase, remove punctuation, tidy spaces) while
# keeping verse/chorus line breaks so structure is preserved.
# Then we take an MD5 fingerprint of each cleaned lyric.
# Two songs with the exact same fingerprint have identical lyrics → keep only one.
df['_lyrics_norm'] = df['lyrics'].astype(str).map(normalize_lyrics)  # clean the lyrics

df['_lyrics_hash'] = df['_lyrics_norm'].map(md5_hash)                 # fingerprint the cleaned lyricsprint(f'Removed duplicate lyrics (hash) : {before - len(df):,}  →  {len(df):,} remain')

before = len(df)df = df.drop_duplicates(subset=['_lyrics_hash']).copy()               # drop lyric duplicates

In [ ]:
# ── Step 9 · Remove songs that are unusually short or absurdly long ───────
# A single clustering algorithm works best when all songs are roughly comparable
# in length.  We use the IQR (interquartile range) rule — a standard statistics
# trick — to define "normal" length:
#   - Find the middle 50% of songs (between the 25th and 75th percentile).
#   - Anything more than 1.5× that spread below the bottom or above the top is an outlier.
# Think of it as trimming the extremely short snippets and extremely long
# repeat-heavy tracks from the dataset.
wc      = df['_word_count']
q1, q3  = wc.quantile(0.25), wc.quantile(0.75)  # bottom and top of the middle 50%

iqr     = q3 - q1                                # the spread of the middle 50%print(f'Removed outliers                : {before - len(df):,}  →  {len(df):,} remain')

lo      = max(1, int(q1 - 1.5 * iqr))           # lower bound (never go below 1 word)print(f'IQR word-count bounds           : [{lo}, {hi}]')

hi      = int(q3 + 1.5 * iqr)                   # upper bounddf = df[(df['_word_count'] >= lo) & (df['_word_count'] <= hi)].copy()
before  = len(df)

In [ ]:
# ── Step 10 · Save the clean dataset ──────────────────────────────────────
# Drop all the temporary helper columns (they start with _) and keep only
# the four columns our feature extraction pipeline will actually use.
# Then save to a new CSV so the raw file is never modified.
final_df = df[['id', 'title', 'artist', 'lyrics']].copy()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)  # create the folder if it does not exist yet
final_df.to_csv(OUT_CSV, index=False)              # index=False → don't write the row numbers into the file

print(f'\nSaved → {OUT_CSV}')final_df.head(5)  # preview the first 5 rows to do a quick sanity check
print(f'Final row count : {len(final_df):,}')